# Week 5 Lab 1: Linear Regression

## Foreword
This lab introduces **Linear Regression**, one of the foundational algorithms in supervised learning. You will learn how to predict continuous values (like house prices) from numerical features.

We explore two approaches:
1. **The Library Way**: Using `scikit-learn` for quick implementation.
2.  **The Math Way**: Implementing Gradient Descent from scratch using only `numpy`.


### Step 1: Import Dependent Packages
We begin by importing the necessary libraries:
*   `LinearRegression` from `sklearn` for the model.
*   `matplotlib` for plotting graphs.
*   `numpy` for numerical operations.

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

### Step 2: Build and Visualize the Dataset
We create a small dataset manually where:
*   `x` represents the House Area.
*   `y` represents the House Price.

We then create a scatter plot to visualize the relationship.

In [ ]:
x = np.array([121, 125, 131, 141, 152, 161]).reshape(-1,1)
y = np.array([300, 350, 425, 405, 496, 517])

plt.scatter(x,y)
plt.xlabel("area")
plt.ylabel("price")
plt.title("House Area vs Price")
plt.show()

### Step 3: Train the Model
We create an instance of the `LinearRegression` model and fit it to our data `(x, y)`.

In [ ]:
# Exercise 1: Initialize the Model
# TODO: Initialize Linear Regression model
model = None

<details>
<summary><strong>Click for Solution</strong></summary>

```python
lr = LinearRegression()
lr.fit(x,y)
```
</details>

### Step 4: Visualize the Model
Now that the model is trained, we can extract the learned parameters:
*   **Slope (w)**: `lr.coef_`
*   **Intercept (b)**: `lr.intercept_`

We then plot the "Best Fit Line" over our original scatter plot.

In [ ]:
w = lr.coef_
b = lr.intercept_
print('Slope: ', w)
print('Intercept: ', b)

plt.scatter(x,y)
plt.xlabel("area")
plt.ylabel("price")
plt.plot([x[0],x[-1]], [x[0]*w+b, x[-1]*w+b])
plt.show()

### Step 5: Make a Prediction
We can now use our trained model to predict the price of a house with a specific area (e.g., 130 sqm).

In [ ]:
testX = np.array([[130]])
prediction = lr.predict(testX)
print(f"Predicted price for area 130: {prediction}")

## Part 2: Linear Regression from Scratch (Gradient Descent)
In this section, we implement the math behind Linear Regression manually using `numpy`.
We will use the dataset `lr2_data.txt`.

### Step 1: Import Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Step 2: Define Gradient Calculation Function
This function implements the mathematical derivation of the gradient:
$$ \frac{\partial J}{\partial \theta} = \frac{1}{m} X^T (X\theta - y) $$

In [ ]:
def generate_gradient(X, theta, y):
    sample_count = X.shape[0]
    return (1./sample_count)*X.T.dot(X.dot(theta)-y)

### Step 3: Define Data Loading Function
This helper function reads the `lr2_data.txt` file and splits it into Features (X) and Labels (y).

In [ ]:
def get_training_data(file_path):
    try:
        orig_data = np.loadtxt(file_path)
        cols = orig_data.shape[1]
        # Last column is y, previous columns are X
        return (orig_data, orig_data[:, :cols - 1], orig_data[:, cols-1:])
    except Exception as e:
        print(f"Error reading data: {e}")
        return None

### Step 4: Initialize Parameters
We initialize our weights vector $\theta$ with ones.

In [ ]:
def init_theta(feature_count):
    return np.ones(feature_count).reshape(feature_count, 1)

### Step 5: Implement Gradient Descent
This is the core training loop:
1.  Calculate current Loss ($J$).
2.  Calculate Gradient.
3.  Update $\theta$ using Learning Rate $\alpha$:
    $$ \theta := \theta - \alpha \times \text{gradient} $$
4.  Repeat until convergence.

In [ ]:
def gradient_descending(X, y, theta, alpha):
    Jthetas= []
    
    # Initial Loss
    Jtheta = (X.dot(theta)-y).T.dot(X.dot(theta)-y)
    index = 0
    gradient = generate_gradient(X, theta, y)
    
    while not np.all(np.absolute(gradient) <= 1e-5):
        theta = theta - alpha * gradient
        gradient = generate_gradient(X, theta, y)
        Jtheta = (X.dot(theta)-y).T.dot(X.dot(theta)-y)
        
        if (index+1) % 10 == 0:
            Jthetas.append((index, Jtheta[0][0]))
        
        index += 1
        if index > 10000: break
            
    return theta, Jthetas

### Step 6 & 7: Visualization Helpers
Define functions to plot the Loss Curve (making sure it goes down!) and the final Regression Line.

In [ ]:
def showJTheta(diff_value):
    p_x = []
    p_y = []
    for (index, sum) in diff_value:
        p_x.append(index)
        p_y.append(sum)
    plt.plot(p_x, p_y, color='b')
    plt.xlabel('steps')
    plt.ylabel('loss function')
    plt.title('Step vs Loss Function Curve')
    plt.show()

def showlinercurve(theta, sample_training_set):
    x, y = sample_training_set[:, 0], sample_training_set[:, 1]
    z = theta[0] + theta[1] * x
    plt.scatter(x, y, color='b', marker='x', label="sample data")
    plt.plot(x, z, 'r', color="r", label="regression curve")
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Linear Regression Curve')
    plt.legend()
    plt.show()

### Step 8: Execution
Now we run our manual implementation.
**Note**: We add a column of `1`s to our Feature Matrix `X`. This is a mathematical trick to handle the Intercept ($b$) within the same matrix multiplication as the Slope ($w$).

In [ ]:
# 1. Read Data
data_path = "lr2_data.txt"
training_data_include_y, training_x, y = get_training_data(data_path)

# 2. Prepare Data (Add Bias Term)
m = len(training_x)
x0 = np.ones((m, 1))
training_x_biased = np.hstack((x0, training_x.reshape(-1, 1)))

# 3. Setup Hyperparameters
sample_count, feature_count = training_x_biased.shape
alpha = 0.0000001

# 4. Train
theta = init_theta(feature_count)
result_theta, Jthetas = gradient_descending(training_x_biased, y.reshape(-1,1), theta, alpha)

# 5. Results
print("w (Bias): {}".format(result_theta[0][0]))
print("w (Slope): {}".format(result_theta[1][0]))

showJTheta(Jthetas)
showlinercurve(result_theta, training_data_include_y)